[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdgordillob/aca_aci_collab/blob/main/notebooks/aca_opt_multi_region_monthly.ipynb)

# aca_opt multi-region monthly anomalies (temperature + wind + precipitation + drought)

Runs `calcular_anomalias_regiones.py`'s existing multi-component orchestrator -- `procesar_anomalias_region()`, already handling temperature (T90/T10), wind power (WP), precipitation (Rx5day), and drought (CDD) together per region -- against the full raw ERA5 archive fetched from Drive, for the regions this pipeline documents as actually populated (national, Antioquia, Cundinamarca-Bogota, Valle del Cauca). No new pipeline logic: this wires existing, already-importable functions to Drive-sourced data across three variables instead of one.

**This is a long run.** Fetching three variables (temperature, precipitation, wind) x 64 years is roughly 3x `aca_opt_full_replication.ipynb`'s ~5.5GB, and Stage 3 runs three anomaly calculations per region. Expect **2.5-3.5 hours total** for the default 4 regions -- test with `YEARS = range(1961, 1965)` and `REGIONS = REGIONS[:1]` first if you just want to confirm it runs before committing to the full range.

**Known pre-existing bug this notebook works around, not fixes:** `calcular_percentil_lluvia.py` saves its output as `era5_lluvia_percentil.nc` (singular), but `calcular_anomalias_lluvia.py` reads `era5_lluvias_percentil.nc` (plural) -- a naming mismatch already visible in this repo's own `data/processed/` (both filenames exist there, side by side). This notebook copies the output under both names rather than silently hiding the inconsistency.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_COLAB

## 1. Get the code

In [ ]:
import sys, os

REPO_ROOT = "/content/aca_indice_climatico_opt"

if IN_COLAB:
    !git clone --depth 1 https://github.com/mdgordillob/aca_indice_climatico_opt.git {REPO_ROOT}
else:
    REPO_ROOT = os.path.abspath("../../aca_indice_climatico_opt-main")  # adjust if running locally

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "scripts"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "utils"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))  # for drive_sync
os.chdir(REPO_ROOT)  # calcular_percentil_lluvia.calcular_percentiles() takes no args -- resolves paths from __file__, needs cwd/repo consistent

## 2. Install dependencies (pinned to requirements.txt)

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y libeccodes-dev > /dev/null
    !pip install -q cfgrib==0.9.14.1 eccodes==2.39.1 rioxarray==0.18.2 geopandas==1.0.1 netCDF4==1.7.2 xarray==2024.11.0

## 3. Mount Drive and fetch everything

All shapefiles (small, ~a few MB total) plus `era5_{tmp,rain,wind}_<year>.grib` for every year in `YEARS`. Uses conventional repo-relative paths throughout (`data/raw/era5/`, `data/processed/...`) rather than a scratch directory -- this is a fresh clone, so there's nothing to collide with, and several of the scripts below (`calcular_percentil_lluvia.calcular_percentiles()`) hardcode those conventional paths rather than accepting overrides.

In [ ]:
import shutil, time

DRIVE_ROOT = "/content/drive/MyDrive/2. Datos"
YEARS = range(1961, 2025)  # narrow this for a quick test run, e.g. range(1961, 1965)

raw_dir = os.path.join(REPO_ROOT, "data", "raw", "era5")
os.makedirs(raw_dir, exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    import drive_sync
    drive_sync.sync(DRIVE_ROOT, REPO_ROOT, only=["shapefiles"])

    completos = os.path.join(DRIVE_ROOT, "era5", "completos")
    t0 = time.perf_counter()
    fetched = {"tmp": 0, "rain": 0, "wind": 0}
    for var in fetched:
        for year in YEARS:
            name = f"era5_{var}_{year}.grib"
            src = os.path.join(completos, name)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(raw_dir, name))
                fetched[var] += 1
    print(f"fetched {fetched} in {time.perf_counter()-t0:.0f}s")
else:
    print("Not in Colab -- assuming data/raw/era5 and data/shapefiles are already populated locally.")

## 4. Stage 1 -- merge/resample the 1961-1990 baseline, all three variables

Only the baseline years, for the same reason as `aca_opt_full_replication.ipynb`: Stage 2 for every variable only ever reads the `1961-1990` slice.

Parallelized over years -- each year writes its own uniquely-named intermediate file (`t2m_daily_{year}.nc` etc.), no cross-year state, unlike drought's CDD (\S6b) -- so this is safely embarrassingly parallel. The first version of this cell ran all 90 year-decodes (3 variables x 30 years) **sequentially**, one at a time -- a plain gap, not something inherent to the problem -- which was most of Stage 1's ~30 minutes.

In [ ]:
import unir_archivos
from multiprocessing import Pool, cpu_count

processed_dir = os.path.join(REPO_ROOT, "data", "processed")
baseline_years = [y for y in range(1961, 1991) if y in YEARS]

# Top-level, named functions -- multiprocessing pickles the task to send
# to worker processes even under Colab's fork start method, and lambdas
# aren't picklable.
def _stage1_task_tmp(args):
    grib_path, year, out_dir = args
    import unir_archivos as u
    u.process_yearly_data_tmp(grib_path, year, "t2m", out_dir)

def _stage1_task_rain(args):
    grib_path, year, out_dir = args
    import unir_archivos as u
    u.process_yearly_precipitation_data(grib_path, year, "tp", out_dir)

def _stage1_task_wind(args):
    grib_path, year, out_dir = args
    import unir_archivos as u
    u.process_yearly_wind_data(grib_path, year, out_dir, ["u10", "v10"])

def run_stage1(var_key, task_fn, output_subdir, merged_name, merge_variable, num_workers=None):
    out_dir = os.path.join(processed_dir, output_subdir)
    os.makedirs(out_dir, exist_ok=True)
    present = [y for y in baseline_years if os.path.exists(os.path.join(raw_dir, f"era5_{var_key}_{y}.grib"))]
    print(f"{var_key}: {len(present)}/{len(baseline_years)} baseline years present")
    t0 = time.perf_counter()
    tasks = [(os.path.join(raw_dir, f"era5_{var_key}_{y}.grib"), y, out_dir) for y in present]
    num_workers = num_workers or max(1, cpu_count() - 1)
    with Pool(num_workers) as pool:
        pool.map(task_fn, tasks)
    unir_archivos.merge_yearly_files(out_dir, os.path.join(processed_dir, merged_name), merge_variable)
    print(f"Stage 1 ({var_key}, {len(present)} years): {(time.perf_counter()-t0)/60:.1f} min")

run_stage1("tmp", _stage1_task_tmp, "daily_by_year_tmp", "era5_daily_combined_tmp.nc", "t2m")
run_stage1("rain", _stage1_task_rain, "daily_by_year_rain", "era5_daily_combined_rain.nc", "tp")
run_stage1("wind", _stage1_task_wind, "daily_by_year_wind", "era5_daily_combined_wind.nc", ["u10", "v10"])

## 5. Stage 2 -- baseline percentiles, all three variables

In [ ]:
import calcular_percentil_temperatura as percentil_tmp
import calcular_percentil_viento as percentil_wind
import calcular_percentil_lluvia as percentil_rain

t0 = time.perf_counter()
est_tmp = percentil_tmp.calcular_percentiles(os.path.join(processed_dir, "era5_daily_combined_tmp.nc"))
percentil_tmp.guardar_percentiles(est_tmp, os.path.join(processed_dir, "era5_temperatura_percentil.nc"), processed_dir, guardar_csv=False)
print(f"temperature percentiles: {(time.perf_counter()-t0)/60:.1f} min")

t0 = time.perf_counter()
est_wind = percentil_wind.calcular_percentiles_viento(os.path.join(processed_dir, "era5_daily_combined_wind.nc"))
percentil_wind.guardar_percentiles_viento(est_wind, os.path.join(processed_dir, "era5_wind_percentil.nc"), processed_dir, guardar_csv=False)
print(f"wind percentiles: {(time.perf_counter()-t0)/60:.1f} min")

t0 = time.perf_counter()
percentil_rain.calcular_percentiles()  # no args -- reads/writes REPO_ROOT/data/processed/ via __file__
# work around the lluvia/lluvias naming mismatch (see intro) rather than hide it:
shutil.copy(
    os.path.join(processed_dir, "era5_lluvia_percentil.nc"),
    os.path.join(processed_dir, "era5_lluvias_percentil.nc"),
)
print(f"rain + drought percentiles: {(time.perf_counter()-t0)/60:.1f} min")

## 6. Stage 3 -- anomalies for every region, all components

`procesar_anomalias_region()` calls temperature/wind/precipitation once **per region**, each re-decoding the same raw grib from scratch -- 4 regions x 3 variables x 64 years = 768 redundant grib-decode passes for data that's identical across regions until the final shapefile clip. Temperature and wind expose a clean load-once (`load_annual_grid_data`/`load_annual_grid_data_safe`, both take an *optional* `shapefile_path`) then clip-per-region split, so the cells below decode each year **once** and clip to all 4 regions from that one decode -- roughly a 4x reduction for those two variables. Precipitation's equivalent (`calcular_anomalias_lluvia.load_grid_data`) requires a shapefile unconditionally with no split available without a more invasive change to that script, so it's left as the original per-region loop for now -- still correct, just not sped up.

**This changes performance only, not the underlying computation** -- same `calcular_anomalias`/`calcular_anomalias_viento` functions, same percentile files, same multiprocessing-over-years pattern (now with each worker handling all 4 regions for its year instead of 1). The open multiprocessing-correctness question from `ARCHITECTURE.pdf` \S9.4 is therefore still open here too -- this notebook does not resolve it, just runs faster either way.

In [ ]:
import calcular_anomalias_temperatura as anomalias_tmp
import calcular_anomalias_viento as anomalias_wind
from multiprocessing import Pool, cpu_count
import xarray as xr

shapefiles_dir = os.path.join(REPO_ROOT, "data", "shapefiles")

REGIONS = [
    {"name": "anomalias_colombia", "shapefile": "colombia_4326.shp"},
    {"name": "anomalias_antioquia", "shapefile": "antioquia_4326.shp"},
    {"name": "anomalias_cundinamarca_bogota", "shapefile": "Cundinamarca_Bogota_4326.shp"},
    {"name": "anomalias_valle_cauca", "shapefile": "valle_cauca_4326.shp"},
    # Shapefiles exist for these too (uncomment to include) -- not part of
    # this repo's documented "actually populated" set, so left off by default:
    # {"name": "anomalias_bogota", "shapefile": "bogota.shp"},
    # {"name": "anomalias_medellin", "shapefile": "medellin_4326.shp"},
    # {"name": "anomalias_cali", "shapefile": "cali_4326.shp"},
    # {"name": "anomalias_san_andres_providencia", "shapefile": "san_andres_providencia.shp"},
]

for region in REGIONS:
    os.makedirs(os.path.join(processed_dir, region["name"]), exist_ok=True)

### 6a. Temperature + wind -- decode once per year, clip to all regions

In [ ]:
def _year_task_temp(args):
    year, grib_path, archivo_percentiles, regions, shapefiles_dir, processed_dir = args
    import calcular_anomalias_temperatura as m
    results = {r["name"]: [] for r in regions}
    try:
        annual = m.load_annual_grid_data(grib_path, year, "t2m", shapefile_path=None)
    except Exception as e:
        print(f"  Error loading tmp {year}: {e}")
        return results
    for region in regions:
        shp = os.path.join(shapefiles_dir, region["shapefile"])
        try:
            shape = m.get_cached_shapefile(shp)
            clipped = annual.rio.write_crs("EPSG:4326", inplace=True)
            clipped = clipped.rio.clip(shape.geometry, shape.crs, drop=True)
        except Exception as e:
            print(f"  Error clipping tmp {year} {region['name']}: {e}")
            continue
        out_dir = os.path.join(processed_dir, region["name"])
        for month in range(1, 13):
            try:
                monthly = m.get_monthly_data(clipped, year, month, "t2m")
                if len(monthly.time) == 0:
                    continue
                ds_month = m.calcular_anomalias(
                    archivo_percentiles, monthly, year, month,
                    os.path.join(out_dir, f"anomalies_temperature_{year}_{month}.nc"),
                    shapefile_path=shp,
                )
                results[region["name"]].append(ds_month.assign_coords(year=year))
            except Exception as e:
                print(f"  Error tmp {region['name']} {year}-{month}: {e}")
    return results

def _year_task_wind(args):
    year, grib_path, archivo_percentiles, regions, shapefiles_dir, processed_dir = args
    import calcular_anomalias_viento as m
    results = {r["name"]: [] for r in regions}
    annual = m.load_annual_grid_data_safe(grib_path, year, "wind_speed", shapefile_path=None)
    if annual is None:
        return results
    for region in regions:
        shp = os.path.join(shapefiles_dir, region["shapefile"])
        try:
            shape = m.get_cached_shapefile(shp)
            clipped = annual.rio.write_crs("EPSG:4326", inplace=True)
            clipped = clipped.rio.clip(shape.geometry, shape.crs, drop=True)
        except Exception as e:
            print(f"  Error clipping wind {year} {region['name']}: {e}")
            continue
        out_dir = os.path.join(processed_dir, region["name"])
        for month in range(1, 13):
            try:
                monthly = m.get_monthly_data(clipped, year, month, "wind_speed")
                if len(monthly.time) == 0:
                    continue
                ds_month = m.calcular_anomalias_viento(
                    archivo_percentiles, monthly, year, month,
                    os.path.join(out_dir, f"anomalies_wind_{year}_{month}.nc"),
                    shapefile_path=shp,
                )
                results[region["name"]].append(ds_month.assign_coords(year=year))
            except Exception as e:
                print(f"  Error wind {region['name']} {year}-{month}: {e}")
    return results

def run_variable_multi_region(var_key, task_fn, archivo_percentiles, output_filename, num_workers=None):
    files = [f for f in os.listdir(raw_dir) if f.endswith(".grib") and var_key in f]
    tasks = []
    for year in YEARS:
        matches = [f for f in files if str(year) in f]
        if len(matches) == 1:
            tasks.append((year, os.path.join(raw_dir, matches[0]), archivo_percentiles, REGIONS, shapefiles_dir, processed_dir))
    print(f"{var_key}: {len(tasks)}/{len(list(YEARS))} years present")

    num_workers = num_workers or max(1, cpu_count() - 1)
    t0 = time.perf_counter()
    with Pool(num_workers) as pool:
        year_results = pool.map(task_fn, tasks)

    combined = {r["name"]: [] for r in REGIONS}
    for yr in year_results:
        for name, items in yr.items():
            combined[name].extend(items)

    for region in REGIONS:
        items = combined[region["name"]]
        if not items:
            print(f"  no {var_key} results for {region['name']}")
            continue
        df = xr.concat(items, dim="time").to_dataframe().reset_index()
        df.to_csv(os.path.join(processed_dir, region["name"], output_filename), index=False)

    print(f"{var_key}: {(time.perf_counter()-t0)/60:.1f} min for all {len(REGIONS)} regions")

run_variable_multi_region("tmp", _year_task_temp, os.path.join(processed_dir, "era5_temperatura_percentil.nc"), "anomalies_temperature_combined.csv")
run_variable_multi_region("wind", _year_task_wind, os.path.join(processed_dir, "era5_wind_percentil.nc"), "anomalies_wind_combined.csv")

### 6b. Precipitation + drought -- parallel across regions, not years

`calcular_anomalias_lluvia.py`'s drought/CDD computation (`calcular_interpolacion`) reads the **previous year's** saved `datos_maximos_{year-1}.nc` to compute the current year's CDD, and writes its own for the next year -- a sequential, year-over-year dependency. Parallelizing *over years* (the same pattern used for temperature/wind above, and `procesar_anomalias_lluvia`'s own `use_multiprocessing=True` default) risks processing years out of order, silently falling back to a cruder approximation for whichever years get unlucky scheduling -- a pre-existing risk in this script's default behavior, not something introduced here.

So instead of decode-once-per-year (unsafe here) or leaving this fully sequential (slow), each **region** gets its own worker process, computing that region's full 1961-2024 precipitation + drought sequentially and correctly inside that one process (year order preserved per region), with the 4 regions running concurrently. Still real parallelism (up to 4x, one region per core), without touching the correctness question above -- each call below runs with `use_multiprocessing=False` internally for exactly that reason.

In [ ]:
def _region_task_rain(args):
    region, shapefiles_dir, processed_dir, raw_dir = args
    import calcular_anomalias_lluvia as m
    shp = os.path.join(shapefiles_dir, region["shapefile"])
    out_dir = os.path.join(processed_dir, region["name"])
    m.procesar_anomalias_lluvia(
        shapefile_path=shp,
        ruta=processed_dir,
        ruta_grib=raw_dir,
        ruta_salida=out_dir,
        use_multiprocessing=False,  # CDD's year-over-year dependency -- must stay sequential within a region
    )
    return region["name"]

t0 = time.perf_counter()
tasks = [(region, shapefiles_dir, processed_dir, raw_dir) for region in REGIONS]
with Pool(min(len(REGIONS), max(1, cpu_count() - 1))) as pool:
    for name in pool.imap_unordered(_region_task_rain, tasks):
        print(f"{name}: done ({(time.perf_counter()-t0)/60:.1f} min elapsed)")
stage3_rain_total = time.perf_counter() - t0
print(f"precipitation + drought, all regions: {stage3_rain_total/60:.1f} min")

## 7. Summary

In [ ]:
import pandas as pd

rows = []
for region in REGIONS:
    region_dir = os.path.join(processed_dir, region["name"])
    for fname in ["anomalies_temperature_combined.csv", "anomalies_wind_combined.csv",
                  "anomalies_precipitation_combined.csv", "anomalies_drought_combined.csv"]:
        path = os.path.join(region_dir, fname)
        rows.append({
            "region": region["name"],
            "file": fname,
            "exists": os.path.exists(path),
            "rows": len(pd.read_csv(path)) if os.path.exists(path) else 0,
        })

summary = pd.DataFrame(rows)
print(f"Precipitation/drought stage total (4 regions in parallel): {stage3_rain_total/60:.1f} min")
summary

## 8. Debug -- the summary above only checks files exist, not that the
## values are right

`DRIVE_ROOT`'s `salidas_colombia.zip` / `salidas_cundinamarca_bogota.zip` are official reference output -- only those two regions have one on Drive (`antioquia`/`valle_cauca` don't), so this section compares where a reference exists and sanity-checks (distribution stats, cross-region distinctness) everywhere else. This is also the actual test of whether the multiprocessing-correctness question from `ARCHITECTURE.pdf` \S9.4 is live here: watch for missing `count_hot`/`count_cold` columns in cell 8b's output.

In [ ]:
import drive_sync

official_dir = os.path.join(processed_dir, "_official_reference")
drive_sync.sync(
    DRIVE_ROOT, REPO_ROOT,
    mapping={
        "salidas_colombia": "data/processed/_official_reference/anomalias_colombia",
        "salidas_cundinamarca_bogota": "data/processed/_official_reference/anomalias_cundinamarca_bogota",
    },
)

for name in ["anomalias_colombia", "anomalias_cundinamarca_bogota"]:
    d = os.path.join(official_dir, name)
    print(name, ":", os.listdir(d) if os.path.isdir(d) else "MISSING")

### 8b. Compare fresh vs. official, per region, per component

In [ ]:
COMPONENT_COLUMNS = {
    "anomalies_temperature_combined.csv": (["year", "month"], ["t_90", "t_10", "count_hot", "count_cold"]),
    "anomalies_wind_combined.csv": (["year", "month"], ["count_above", "anomalies_above"]),
    "anomalies_precipitation_combined.csv": (["Año", "Mes"], ["Anomalia_Lluvia", "Rx5day"]),
    "anomalies_drought_combined.csv": (["Año", "Mes"], ["Anomalia_Sequia", "CDD"]),
}

for region_name in ["anomalias_colombia", "anomalias_cundinamarca_bogota"]:
    official_region_dir = os.path.join(official_dir, region_name)
    mine_region_dir = os.path.join(processed_dir, region_name)
    if not os.path.isdir(official_region_dir):
        print(f"no official reference for {region_name}, skipping")
        continue
    print(f"\n=== {region_name} ===")
    for fname, (keys, cols) in COMPONENT_COLUMNS.items():
        official_path = os.path.join(official_region_dir, fname)
        mine_path = os.path.join(mine_region_dir, fname)
        if not (os.path.exists(official_path) and os.path.exists(mine_path)):
            print(f"  {fname}: missing on one side (official={os.path.exists(official_path)}, mine={os.path.exists(mine_path)})")
            continue
        official_df = pd.read_csv(official_path)
        mine_df = pd.read_csv(mine_path)
        present_cols = [c for c in cols if c in official_df.columns and c in mine_df.columns]
        missing = [c for c in cols if c not in present_cols]
        if missing:
            print(f"  {fname}: WARNING missing columns {missing}")
        merged = mine_df.merge(official_df, on=keys, suffixes=("_mine", "_official"))
        print(f"  {fname}: {len(merged)}/{len(official_df)} rows matched by {keys}")
        for c in present_cols:
            diff = (merged[f"{c}_mine"] - merged[f"{c}_official"]).abs()
            print(f"    {c}: max diff = {diff.max():.3e}, mean diff = {diff.mean():.3e}")

### 8c. Sanity stats for regions with no official reference (`antioquia`, `valle_cauca`)

In [ ]:
for region_name in ["anomalias_antioquia", "anomalias_valle_cauca"]:
    print(f"\n=== {region_name} (no official reference -- sanity check only) ===")
    for fname, (keys, cols) in COMPONENT_COLUMNS.items():
        path = os.path.join(processed_dir, region_name, fname)
        if not os.path.exists(path):
            print(f"  {fname}: MISSING")
            continue
        df = pd.read_csv(path)
        for c in cols:
            if c not in df.columns:
                print(f"  {fname}: column {c!r} MISSING (have {df.columns.tolist()})")
                continue
            print(f"  {fname}.{c}: n_non_na={df[c].notna().sum()}/{len(df)}, mean={df[c].mean():.4f}, std={df[c].std():.4f}, min={df[c].min():.4f}, max={df[c].max():.4f}")

### 8d. Confirm regions actually differ

Catches a shapefile clip silently no-op'ing (e.g. every region accidentally getting the national extent).

In [ ]:
print("\n=== cross-region distinctness (temperature t_90) ===")
temp_by_region = {}
for region in REGIONS:
    path = os.path.join(processed_dir, region["name"], "anomalies_temperature_combined.csv")
    if os.path.exists(path):
        temp_by_region[region["name"]] = pd.read_csv(path).sort_values(["year", "month"])["t_90"].reset_index(drop=True)

import itertools
for a, b in itertools.combinations(temp_by_region, 2):
    identical = temp_by_region[a].equals(temp_by_region[b])
    corr = temp_by_region[a].corr(temp_by_region[b])
    flag = "  <-- SUSPICIOUS, investigate" if identical else ""
    print(f"{a} vs {b}: identical={identical}, correlation={corr:.4f}{flag}")